In [2]:
# synthetic_map.py

from astropy.io import fits
from astropy.wcs import WCS
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm
import numpy as np
import reproject


def get_hdu_bounds(header: fits.Header) -> tuple[int]:
    """Get the bounds of an ImageHDU header in terms of RA and dec.
    
    :params fits.Header input_header: The header to get the bounds of
    :returns tuple[int]: Returns the int tuple (x1, y1, x2, y2) where x1 <= x2 and y1 <= y2
    """

    # get dimensions of the boundary of the input header
    iw, ih = header["NAXIS1"], header["NAXIS2"]
    
    # coords are arranged top row, bot row, left col, right col, and top to bot and left to right within them
    x_coords = np.concat([np.arange(iw), np.arange(iw), np.zeros(shape=(ih,)), np.repeat(iw - 1, ih)])
    y_coords = np.concat([np.zeros(shape=(iw,)), np.repeat(ih - 1, iw), np.arange(ih), np.arange(ih)])
    
    # convert input pixel coordinates to wcs space
    input_wcs = WCS(header)
    boundary_wcs_x, boundary_wcs_y = input_wcs.pixel_to_world_values(x_coords, y_coords)

    x1 = int(np.floor(np.min(boundary_wcs_x)))
    x2 = int(np.ceil(np.max(boundary_wcs_x)))
    y1 = int(np.floor(np.min(boundary_wcs_y)))
    y2 = int(np.ceil(np.max(boundary_wcs_y)))

    return (x1, y1, x2, y2)

In [3]:
hdul = fits.open('/data1/thomasli/selfcal/outputs/nep_det1_6p2arcsec/mosaic/nep_6p2arcsec_det1_ch15.fits')

In [6]:
header = hdul[0].header

In [ ]:
303/252

In [7]:
get_hdu_bounds(header)

(252, 59, 303, 75)

In [16]:
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u


def compute_extent(header):
    # Load header file
    wcs = WCS(header)
    nx, ny = header['NAXIS1'], header['NAXIS2']

    # Compute midpoint pixels on each edge
    x_center = (nx - 1) / 2
    y_center = (ny - 1) / 2

    edge_pixels = {
        "left":    (0,        y_center),
        "right":   (nx - 1,   y_center),
        "bottom":  (x_center, 0),
        "top":     (x_center, ny - 1),
    }

    # Convert to world coordinates
    edge_coords = {}
    for key, (x, y) in edge_pixels.items():
        ra_dec = wcs.pixel_to_world(x, y)
        edge_coords[key] = SkyCoord(ra_dec)

    # Compute angular extents in radians
    x_extent = edge_coords["left"].separation(edge_coords["right"]).to(u.rad).value
    y_extent = edge_coords["bottom"].separation(edge_coords["top"]).to(u.rad).value

    return x_extent, y_extent

In [17]:
compute_extent(header)

(np.float64(0.2550067244120153), np.float64(0.23822109869709218))